# F1 Race Performance Analytics — Data Preparation & Exploration

This notebook handles the first complete data stage of the project:

- Load Australia, China and Japan race data
- Apply one reusable cleaning pipeline
- Validate the three datasets
- Explore driver, team, tyre and stint data
- Combine the three races
- Save analysis-ready CSV files

We are deliberately keeping this as **one notebook** instead of creating a separate cleaning notebook.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Libraries loaded successfully.")


## 1. Project paths


In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

RACES = [
    "Australian Grand Prix",
    "Chinese Grand Prix",
    "Japanese Grand Prix"
]

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA)
print("Races:", RACES)


## 2. Reusable race-loading and cleaning pipeline


In [ ]:
NUMERIC_COLUMNS = [
    "time", "lap", "stint",
    "s1", "s2", "s3",
    "ms1", "ms2", "ms3",
    "life", "pos"
]

def load_and_clean_race(race_name):
    """Load and clean the session_laptimes.json for one race."""

    file_path = (
        RAW_DATA
        / race_name
        / "Race"
        / "session_laptimes.json"
    )

    if not file_path.exists():
        raise FileNotFoundError(
            f"Could not find: {file_path}"
        )

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # The repository's session_laptimes.json is a
    # dictionary of equal-length column arrays.
    df = pd.DataFrame(data)

    # Convert numeric fields safely.
    for column in NUMERIC_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

    # Standardize useful text fields.
    for column in ["drv", "team", "compound"]:
        if column in df.columns:
            df[column] = df[column].astype("string").str.strip()

    # Add race identifier.
    df["race"] = race_name

    # Remove exact duplicate rows.
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    duplicates_removed = before - len(df)

    print(
        f"{race_name}: "
        f"{len(df):,} rows | "
        f"{len(df.columns)} columns | "
        f"{df['drv'].nunique()} drivers | "
        f"{duplicates_removed} duplicate rows removed"
    )

    return df


## 3. Load and clean all three races


In [ ]:
race_dfs = {}

for race in RACES:
    race_dfs[race] = load_and_clean_race(race)

australia_df = race_dfs["Australian Grand Prix"]
china_df = race_dfs["Chinese Grand Prix"]
japan_df = race_dfs["Japanese Grand Prix"]


## 4. Validate that the three races have compatible schemas


In [ ]:
schema_summary = pd.DataFrame({
    race: sorted(df.columns)
    for race, df in race_dfs.items()
})

print("Column count by race:")

for race, df in race_dfs.items():
    print(f"{race}: {len(df.columns)} columns")

common_columns = set(australia_df.columns)

for df in [china_df, japan_df]:
    common_columns &= set(df.columns)

print("\nCommon columns:")
print(sorted(common_columns))

for race, df in race_dfs.items():
    missing_common = sorted(common_columns - set(df.columns))

    if missing_common:
        print(f"{race} missing:", missing_common)
    else:
        print(f"{race}: schema compatible ✓")


## 5. Combined three-race dataset


In [ ]:
all_races_df = pd.concat(
    [australia_df, china_df, japan_df],
    ignore_index=True,
    sort=False
)

print("Combined shape:", all_races_df.shape)
display(all_races_df.head())


In [ ]:
print("Rows by race:")
display(
    all_races_df["race"]
    .value_counts()
    .rename_axis("race")
    .reset_index(name="rows")
)

print("\nDrivers by race:")

drivers_by_race = (
    all_races_df
    .groupby("race")["drv"]
    .nunique()
    .reset_index(name="drivers")
)

display(drivers_by_race)


## 6. Data-quality checks


In [ ]:
quality = []

for race, df in race_dfs.items():
    quality.append({
        "race": race,
        "rows": len(df),
        "columns": len(df.columns),
        "drivers": df["drv"].nunique(),
        "teams": df["team"].nunique(),
        "duplicate_rows": df.duplicated().sum(),
        "missing_lap_time": df["time"].isna().sum(),
        "missing_driver": df["drv"].isna().sum(),
        "missing_team": df["team"].isna().sum(),
        "missing_compound": df["compound"].isna().sum()
    })

quality_df = pd.DataFrame(quality)
display(quality_df)


## 7. Driver performance across all three races


In [ ]:
valid_laps = all_races_df.dropna(
    subset=["time", "drv", "team"]
).copy()

driver_performance = (
    valid_laps
    .groupby(["drv", "team"])
    .agg(
        races=("race", "nunique"),
        laps=("lap", "count"),
        best_lap=("time", "min"),
        average_lap=("time", "mean"),
        median_lap=("time", "median"),
        lap_time_std=("time", "std")
    )
    .reset_index()
    .sort_values("average_lap")
)

display(driver_performance)


## 8. Performance by race


In [ ]:
race_performance = (
    valid_laps
    .groupby(["race", "drv", "team"])
    .agg(
        laps=("lap", "count"),
        best_lap=("time", "min"),
        average_lap=("time", "mean"),
        median_lap=("time", "median"),
        lap_time_std=("time", "std")
    )
    .reset_index()
)

display(
    race_performance
    .sort_values(["race", "average_lap"])
)


## 9. Tyre compound analysis


In [ ]:
compound_performance = (
    valid_laps
    .dropna(subset=["compound"])
    .groupby(["race", "compound"])
    .agg(
        laps=("lap", "count"),
        drivers=("drv", "nunique"),
        average_lap=("time", "mean"),
        median_lap=("time", "median"),
        best_lap=("time", "min")
    )
    .reset_index()
)

display(compound_performance)


## 10. Stint analysis


In [ ]:
stint_performance = (
    valid_laps
    .dropna(subset=["stint", "compound"])
    .groupby(["race", "drv", "stint", "compound"])
    .agg(
        laps=("lap", "count"),
        average_lap=("time", "mean"),
        best_lap=("time", "min"),
        average_tyre_life=("life", "mean")
    )
    .reset_index()
)

display(stint_performance.head(50))


## 11. Tyre-age / degradation analysis


In [ ]:
degradation = (
    valid_laps
    .dropna(subset=["life"])
    .groupby(["race", "compound", "life"])
    .agg(
        average_lap=("time", "mean"),
        median_lap=("time", "median"),
        observations=("time", "count")
    )
    .reset_index()
)

display(degradation.head(30))


In [ ]:
# Show the tyre-age relationship for the most common compound.
if not degradation.empty:
    common_compound = (
        valid_laps["compound"]
        .value_counts()
        .index[0]
    )

    plot_df = degradation[
        degradation["compound"] == common_compound
    ]

    plt.figure(figsize=(11, 6))

    for race in RACES:
        temp = plot_df[plot_df["race"] == race]

        if not temp.empty:
            plt.plot(
                temp["life"],
                temp["average_lap"],
                marker="o",
                label=race
            )

    plt.xlabel("Tyre Life")
    plt.ylabel("Average Lap Time (seconds)")
    plt.title(
        f"Lap-Time Trend vs Tyre Life — {common_compound}"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


## 12. Driver consistency


In [ ]:
driver_consistency = (
    valid_laps
    .groupby(["race", "drv", "team"])
    .agg(
        average_lap=("time", "mean"),
        lap_time_std=("time", "std"),
        laps=("time", "count")
    )
    .reset_index()
    .sort_values(["race", "lap_time_std"])
)

display(driver_consistency)


## 13. Save processed datasets


In [ ]:
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

for race, df in race_dfs.items():
    filename = (
        race.lower()
        .replace(" grand prix", "")
        .replace(" ", "_")
        + "_race_laptimes.csv"
    )

    output_path = PROCESSED_DATA / filename
    df.to_csv(output_path, index=False)

    print("Saved:", output_path)

combined_path = PROCESSED_DATA / "f1_2026_three_races.csv"
all_races_df.to_csv(combined_path, index=False)

print("\nSaved combined dataset:")
print(combined_path)


## 14. Final project-ready dataset summary


In [ ]:
print("=" * 65)
print("F1 2026 — THREE-RACE DATASET")
print("=" * 65)

print(f"Total rows: {len(all_races_df):,}")
print(f"Total columns: {len(all_races_df.columns)}")
print(f"Races: {all_races_df['race'].nunique()}")
print(f"Drivers: {all_races_df['drv'].nunique()}")
print(f"Teams: {all_races_df['team'].nunique()}")

print("\nRows per race:")
print(all_races_df["race"].value_counts())

print("\nMissing lap times:")
print(all_races_df["time"].isna().sum())

print("\nExact duplicate rows:")
print(all_races_df.duplicated().sum())
